# Phase 1 A/B: does `media_resolution` / `thinking_config` move dev NED?

Throwaway experiment for `docs/superpowers/plans/2026-08-21-transcription-quality-push.md`
(deficiency **D1**): `src.gemini_transcribe.call_gemini_once` has never set
`media_resolution` or `thinking_config`, so every Gemini call this project has
ever made ran at default (downsampled) image resolution with thinking off --
on a task that is entirely fine-detail handwriting reading.

Three arms over the same 197 dev card crops, **greedy (`temperature=0`, one
sample)** so the only thing varying is the generation config:

| arm | media_resolution | thinking |
|---|---|---|
| `baseline` | default | off |
| `high` | `MEDIA_RESOLUTION_HIGH` | off |
| `high_thinking` | `MEDIA_RESOLUTION_HIGH` | `low` |

591 requests total -- **but each flash-lite model's free-tier budget is ~500/day**, so one session covers roughly two arms; the third resumes the next day (this notebook is resumable per arm). Scoring happens locally afterwards (ground truth is not on
Kaggle) -- download the three CSVs from this notebook's output.

Inputs to attach: `museumscat-card-crops`, `museumscat-gemini-code`.
Secret required: `GEMINI_API_KEY` (the Kaggle-side key, independent of the local one).

## Cell 1: locate the attached inputs

In [ ]:
import glob
import os


def _find_dataset_dir(slug):
    candidates = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input (checked: {candidates})")


CROPS_DIR = _find_dataset_dir("museumscat-card-crops")
CODE_DIR = _find_dataset_dir("museumscat-gemini-code")
print("card-crops dataset:", CROPS_DIR, os.listdir(CROPS_DIR))
print("code dataset:", CODE_DIR, sorted(os.listdir(CODE_DIR))[:5], "...")

## Cell 2: dependency + API key

In [ ]:
!pip install -q google-genai

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))

## Cell 3: unpack the code and the crop images

Same wiring as `kaggle_run_gemini_test.ipynb` Cells 4-6: `museumscat-gemini-code`
uploads a single `src.zip` which Kaggle flattens, so `CODE_DIR` holds the `.py`
files directly; the 877MB crops dataset is symlinked read-only.

In [ ]:
import shutil
import zipfile

_SCRATCH = "/kaggle/working/_crops_scratch"


def locate(name):
    """Absolute path to CROPS_DIR's `name` entry, extracting `name.zip` into
    _SCRATCH first if Kaggle left it zipped instead of auto-extracting it."""
    as_dir = os.path.join(CROPS_DIR, name)
    if os.path.isdir(as_dir):
        return as_dir
    extracted = os.path.join(_SCRATCH, name)
    if not os.path.isdir(extracted):
        with zipfile.ZipFile(os.path.join(CROPS_DIR, f"{name}.zip")) as zf:
            zf.extractall(os.path.join(_SCRATCH, f"_{name}_unzip"))
        shutil.move(os.path.join(_SCRATCH, f"_{name}_unzip", name), extracted)
    return extracted


CARD_CROPS_SRC = locate("card_crops")
CROP_IMAGES = os.path.join(CARD_CROPS_SRC, "images")
print("crop images:", CROP_IMAGES, len(os.listdir(CROP_IMAGES)), "files")

REPO_DIR = "/kaggle/working/repo"
os.makedirs(os.path.join(REPO_DIR, "src"), exist_ok=True)
for py_file in glob.glob(os.path.join(CODE_DIR, "*.py")):
    shutil.copyfile(py_file, os.path.join(REPO_DIR, "src", os.path.basename(py_file)))
os.chdir(REPO_DIR)
print(os.getcwd(), "->", len(os.listdir("src")), "modules")

## Cell 4: the 197 dev card crops

Inlined rather than pulled from a dataset: this is exactly the crop set behind
`data/processed/gemini/dev_card_predictions.csv` (the relevance-filtered dev
cards the shipped pipeline transcribed), so the arms are directly comparable to
the shipped numbers, and inlining avoids re-versioning a dataset for a
throwaway experiment.

In [ ]:
DEV_CROPS = [["1709789_card00.jpeg","1709789.jpeg"],["1706390_card00.jpeg","1706390.jpeg"],["1707822_card01.jpeg","1707822.jpeg"],["1707950_card01.jpeg","1707950.jpeg"],["1728566_card01.jpeg","1728566.jpeg"],["1706758_card00.jpeg","1706758.jpeg"],["1706758_card01.jpeg","1706758.jpeg"],["1744476_card01.jpeg","1744476.jpeg"],["1709619_card00.jpeg","1709619.jpeg"],["1743709_card00.jpeg","1743709.jpeg"],["1709202_card00.jpeg","1709202.jpeg"],["1707214_card01.jpeg","1707214.jpeg"],["1709129_card00.jpeg","1709129.jpeg"],["1705468_card01.jpeg","1705468.jpeg"],["1728453_card01.jpeg","1728453.jpeg"],["1743561_card01.jpeg","1743561.jpeg"],["1728574_card01.jpeg","1728574.jpeg"],["1707996_card00.jpeg","1707996.jpeg"],["1706016_card00.jpeg","1706016.jpeg"],["1709276_card01.jpeg","1709276.jpeg"],["1706094_card00.jpeg","1706094.jpeg"],["1707961_card01.jpeg","1707961.jpeg"],["1708663_card00.jpeg","1708663.jpeg"],["1706465_card00.jpeg","1706465.jpeg"],["1706465_card01.jpeg","1706465.jpeg"],["1705986_card00.jpeg","1705986.jpeg"],["1707816_card01.jpeg","1707816.jpeg"],["1742772_card00.jpeg","1742772.jpeg"],["1742772_card02.jpeg","1742772.jpeg"],["1707546_card00.jpeg","1707546.jpeg"],["1706481_card01.jpeg","1706481.jpeg"],["1707429_card01.jpeg","1707429.jpeg"],["1710346_card00.jpeg","1710346.jpeg"],["1743921_card00.jpeg","1743921.jpeg"],["1705416_card00.jpeg","1705416.jpeg"],["1709989_card00.jpeg","1709989.jpeg"],["1709989_card01.jpeg","1709989.jpeg"],["1709989_card02.jpeg","1709989.jpeg"],["1709989_card03.jpeg","1709989.jpeg"],["1709989_card04.jpeg","1709989.jpeg"],["1705903_card00.jpeg","1705903.jpeg"],["1705903_card01.jpeg","1705903.jpeg"],["1707271_card00.jpeg","1707271.jpeg"],["1706815_card00.jpeg","1706815.jpeg"],["1706815_card01.jpeg","1706815.jpeg"],["1743707_card01.jpeg","1743707.jpeg"],["1718185_card01.jpeg","1718185.jpeg"],["1709175_card00.jpeg","1709175.jpeg"],["1706979_card00.jpeg","1706979.jpeg"],["1706979_card01.jpeg","1706979.jpeg"],["1706979_card02.jpeg","1706979.jpeg"],["1709457_card01.jpeg","1709457.jpeg"],["1708912_card00.jpeg","1708912.jpeg"],["1705778_card00.jpeg","1705778.jpeg"],["1705485_card00.jpeg","1705485.jpeg"],["1705485_card02.jpeg","1705485.jpeg"],["1707739_card00.jpeg","1707739.jpeg"],["1709200_card00.jpeg","1709200.jpeg"],["1706766_card00.jpeg","1706766.jpeg"],["1706766_card01.jpeg","1706766.jpeg"],["1744456_card00.jpeg","1744456.jpeg"],["1709845_card00.jpeg","1709845.jpeg"],["1709796_card00.jpeg","1709796.jpeg"],["1708464_card00.jpeg","1708464.jpeg"],["1707932_card01.jpeg","1707932.jpeg"],["1707430_card01.jpeg","1707430.jpeg"],["1705930_card00.jpeg","1705930.jpeg"],["1706478_card00.jpeg","1706478.jpeg"],["1706478_card02.jpeg","1706478.jpeg"],["1708800_card00.jpeg","1708800.jpeg"],["1709794_card00.jpeg","1709794.jpeg"],["1707396_card01.jpeg","1707396.jpeg"],["1743896_card00.jpeg","1743896.jpeg"],["1706065_card00.jpeg","1706065.jpeg"],["1705830_card00.jpeg","1705830.jpeg"],["1705830_card02.jpeg","1705830.jpeg"],["1709105_card00.jpeg","1709105.jpeg"],["1705589_card00.jpeg","1705589.jpeg"],["1709061_card00.jpeg","1709061.jpeg"],["1707087_card00.jpeg","1707087.jpeg"],["1743021_card00.jpeg","1743021.jpeg"],["1705940_card02.jpeg","1705940.jpeg"],["1705940_card03.jpeg","1705940.jpeg"],["1709382_card00.jpeg","1709382.jpeg"],["1744458_card01.jpeg","1744458.jpeg"],["1709667_card01.jpeg","1709667.jpeg"],["1706911_card01.jpeg","1706911.jpeg"],["1707285_card00.jpeg","1707285.jpeg"],["1707285_card01.jpeg","1707285.jpeg"],["1707285_card02.jpeg","1707285.jpeg"],["1705463_card01.jpeg","1705463.jpeg"],["1744675_card01.jpeg","1744675.jpeg"],["1707539_card00.jpeg","1707539.jpeg"],["1709028_card00.jpeg","1709028.jpeg"],["1707761_card00.jpeg","1707761.jpeg"],["1706292_card00.jpeg","1706292.jpeg"],["1706965_card00.jpeg","1706965.jpeg"],["1744642_card00.jpeg","1744642.jpeg"],["1708936_card01.jpeg","1708936.jpeg"],["1707856_card00.jpeg","1707856.jpeg"],["1706982_card00.jpeg","1706982.jpeg"],["1706929_card01.jpeg","1706929.jpeg"],["1743813_card00.jpeg","1743813.jpeg"],["1742723_card00.jpeg","1742723.jpeg"],["1743912_card00.jpeg","1743912.jpeg"],["1743912_card01.jpeg","1743912.jpeg"],["1705668_card01.jpeg","1705668.jpeg"],["1707673_card01.jpeg","1707673.jpeg"],["1708885_card00.jpeg","1708885.jpeg"],["1743538_card00.jpeg","1743538.jpeg"],["1705625_card00.jpeg","1705625.jpeg"],["1705667_card01.jpeg","1705667.jpeg"],["1707084_card00.jpeg","1707084.jpeg"],["1706030_card01.jpeg","1706030.jpeg"],["1706030_card02.jpeg","1706030.jpeg"],["1709431_card00.jpeg","1709431.jpeg"],["1709431_card01.jpeg","1709431.jpeg"],["1709431_card02.jpeg","1709431.jpeg"],["1743537_card01.jpeg","1743537.jpeg"],["1743533_card01.jpeg","1743533.jpeg"],["1709033_card00.jpeg","1709033.jpeg"],["1709033_card01.jpeg","1709033.jpeg"],["1707818_card00.jpeg","1707818.jpeg"],["1707818_card02.jpeg","1707818.jpeg"],["1705472_card01.jpeg","1705472.jpeg"],["1743909_card00.jpeg","1743909.jpeg"],["1743909_card01.jpeg","1743909.jpeg"],["1705819_card01.jpeg","1705819.jpeg"],["1706933_card01.jpeg","1706933.jpeg"],["1705950_card00.jpeg","1705950.jpeg"],["1705950_card01.jpeg","1705950.jpeg"],["1705950_card02.jpeg","1705950.jpeg"],["1705950_card03.jpeg","1705950.jpeg"],["1705950_card04.jpeg","1705950.jpeg"],["1705950_card05.jpeg","1705950.jpeg"],["1705950_card06.jpeg","1705950.jpeg"],["1706768_card00.jpeg","1706768.jpeg"],["1706496_card00.jpeg","1706496.jpeg"],["1705665_card01.jpeg","1705665.jpeg"],["1706468_card00.jpeg","1706468.jpeg"],["1706468_card01.jpeg","1706468.jpeg"],["1744625_card00.jpeg","1744625.jpeg"],["1744625_card02.jpeg","1744625.jpeg"],["1708370_card00.jpeg","1708370.jpeg"],["1706878_card00.jpeg","1706878.jpeg"],["1743988_card00.jpeg","1743988.jpeg"],["1743988_card01.jpeg","1743988.jpeg"],["1743496_card02.jpeg","1743496.jpeg"],["1743949_card00.jpeg","1743949.jpeg"],["1706347_card00.jpeg","1706347.jpeg"],["1706347_card02.jpeg","1706347.jpeg"],["1707507_card00.jpeg","1707507.jpeg"],["1709178_card00.jpeg","1709178.jpeg"],["1709178_card01.jpeg","1709178.jpeg"],["1705571_card01.jpeg","1705571.jpeg"],["1709994_card01.jpeg","1709994.jpeg"],["1709500_card01.jpeg","1709500.jpeg"],["1705578_card01.jpeg","1705578.jpeg"],["1707616_card00.jpeg","1707616.jpeg"],["1707440_card01.jpeg","1707440.jpeg"],["1707535_card00.jpeg","1707535.jpeg"],["1707535_card01.jpeg","1707535.jpeg"],["1707983_card01.jpeg","1707983.jpeg"],["1744698_card01.jpeg","1744698.jpeg"],["1708784_card00.jpeg","1708784.jpeg"],["1743795_card00.jpeg","1743795.jpeg"],["1709800_card00.jpeg","1709800.jpeg"],["1707063_card00.jpeg","1707063.jpeg"],["1706276_card00.jpeg","1706276.jpeg"],["1706276_card02.jpeg","1706276.jpeg"],["1705431_card01.jpeg","1705431.jpeg"],["1743543_card01.jpeg","1743543.jpeg"],["1728593_card00.jpeg","1728593.jpeg"],["1707875_card00.jpeg","1707875.jpeg"],["1707765_card01.jpeg","1707765.jpeg"],["1709217_card00.jpeg","1709217.jpeg"],["1706862_card00.jpeg","1706862.jpeg"],["1706862_card01.jpeg","1706862.jpeg"],["1706306_card00.jpeg","1706306.jpeg"],["1706306_card01.jpeg","1706306.jpeg"],["1706306_card02.jpeg","1706306.jpeg"],["1706306_card03.jpeg","1706306.jpeg"],["1709985_card00.jpeg","1709985.jpeg"],["1707307_card01.jpeg","1707307.jpeg"],["1705780_card01.jpeg","1705780.jpeg"],["1706762_card00.jpeg","1706762.jpeg"],["1706951_card01.jpeg","1706951.jpeg"],["1705988_card00.jpeg","1705988.jpeg"],["1743779_card01.jpeg","1743779.jpeg"],["1709471_card01.jpeg","1709471.jpeg"],["1705740_card01.jpeg","1705740.jpeg"],["1705740_card02.jpeg","1705740.jpeg"],["1743505_card01.jpeg","1743505.jpeg"],["1709787_card00.jpeg","1709787.jpeg"],["1743037_card02.jpeg","1743037.jpeg"],["1709937_card01.jpeg","1709937.jpeg"],["1705536_card00.jpeg","1705536.jpeg"]]
print(len(DEV_CROPS), 'dev crops')

## Cell 5: the three arms

`PROMPT` and `RESPONSE_SCHEMA` are imported from the code dataset, not
re-typed, so the arms differ from production *only* in `GenerateContentConfig`.

In [ ]:
from google.genai import types

from src.gemini_batch_runner import is_daily_quota_exhausted
from src.gemini_transcribe import (PROMPT, RESPONSE_SCHEMA, RateLimiter, build_client,
                                    _parse_response_json)

MODEL = "gemini-3.5-flash-lite"

ARMS = {
    "baseline": dict(media_resolution=None, thinking=None),
    "high": dict(media_resolution="MEDIA_RESOLUTION_HIGH", thinking=None),
    "high_thinking": dict(media_resolution="MEDIA_RESOLUTION_HIGH", thinking="low"),
}


def make_config(arm):
    spec = ARMS[arm]
    kw = dict(system_instruction=PROMPT, temperature=0.0,
              response_mime_type="application/json", response_json_schema=RESPONSE_SCHEMA)
    if spec["media_resolution"]:
        kw["media_resolution"] = getattr(types.MediaResolution, spec["media_resolution"])
    if spec["thinking"]:
        kw["thinking_config"] = types.ThinkingConfig(thinking_level=spec["thinking"])
    return types.GenerateContentConfig(**kw)


for arm in ARMS:
    make_config(arm)
print("all arm configs construct OK:", list(ARMS))

## Cell 6: run one arm

Resumable the same way `src.gemini_batch_runner` is -- re-running skips any
`crop_filename` already in the arm's output CSV, so a session that dies partway
picks up where it left off. Output schema matches
`dev_card_predictions.csv` so the result can be scored locally through the real
production merge (`src.analyze_rank_headroom.build_dev_context(predictions_path=...)`).

In [ ]:
import csv
import json
import time

OUT_DIR = "/kaggle/working"


def run_arm(arm, rpm=12, limit=None):
    out = os.path.join(OUT_DIR, f"dev_arm_{arm}.csv")
    done = set()
    if os.path.exists(out):
        with open(out) as f:
            done = {r["crop_filename"] for r in csv.DictReader(f)}
    pending = [(c, s) for c, s in DEV_CROPS if c not in done]
    if limit:
        pending = pending[:limit]
    print(f"[{arm}] {len(pending)} pending of {len(DEV_CROPS)}", flush=True)

    client = build_client(os.environ["GEMINI_API_KEY"])
    limiter = RateLimiter(rpm)
    cfg = make_config(arm)

    is_new = not os.path.exists(out)
    with open(out, "a", newline="") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(["crop_filename", "source_image_file", "verbatimDate",
                        "verbatimDate_confidence", "verbatimLocality",
                        "verbatimLocality_confidence", "models_used", "raw_samples_json"])
            f.flush()
        fails = 0
        for i, (crop, source) in enumerate(pending):
            with open(os.path.join(CROP_IMAGES, crop), "rb") as fh:
                data = fh.read()
            parsed = None
            exhausted = False
            for attempt in range(4):
                try:
                    limiter.wait()
                    resp = client.models.generate_content(
                        model=MODEL,
                        contents=[{"inline_data": {"data": data, "mimeType": "image/jpeg"}}],
                        config=cfg)
                    parsed = _parse_response_json(resp.text)
                    break
                except Exception as e:
                    # A *daily* quota 429 will not clear by retrying -- stop the
                    # arm rather than burn ~70s per remaining crop failing four
                    # times each (this cost ~1.5h of session on the first run).
                    # Per-minute 429s and transport errors still retry.
                    if is_daily_quota_exhausted(e):
                        print(f"  [{arm}] daily quota exhausted at {crop} -- "
                              f"stopping this arm, resume it in a later session", flush=True)
                        exhausted = True
                        break
                    msg = str(e)
                    if attempt == 3:
                        print(f"  !! {crop}: {msg[:160]}", flush=True)
                        fails += 1
                        break
                    time.sleep(20 if "429" in msg else 5)
            if exhausted:
                break
            if parsed is None:
                continue
            w.writerow([crop, source, parsed["verbatimDate"], 1.0,
                        parsed["verbatimLocality"], 1.0, MODEL,
                        json.dumps([{**parsed, "_model": MODEL}])])
            f.flush()
            if (i + 1) % 25 == 0:
                print(f"  [{arm}] {i + 1}/{len(pending)}", flush=True)
    print(f"[{arm}] done, {fails} failures -> {out}", flush=True)


run_arm("baseline")

## Cell 7: the other two arms

In [ ]:
run_arm("high")

In [ ]:
run_arm("high_thinking")

## Cell 8: summary + how to collect

No ground truth on Kaggle, so this only reports raw agreement between arms --
the actual NED comparison runs locally. **Save Version** before closing, then:

```
kaggle kernels output matiaspoullain/museumscat-phase1-media-resolution -p <dir>
```

In [ ]:
import collections

loaded = {}
for arm in ARMS:
    p = os.path.join(OUT_DIR, f"dev_arm_{arm}.csv")
    if os.path.exists(p):
        with open(p) as f:
            loaded[arm] = {r["crop_filename"]: (r["verbatimDate"], r["verbatimLocality"])
                           for r in csv.DictReader(f)}
        print(f"{arm}: {len(loaded[arm])} rows")

if "baseline" in loaded:
    for arm in ("high", "high_thinking"):
        if arm not in loaded:
            continue
        shared = set(loaded["baseline"]) & set(loaded[arm])
        diff = collections.Counter()
        for c in shared:
            for k, field in enumerate(("verbatimDate", "verbatimLocality")):
                if loaded["baseline"][c][k] != loaded[arm][c][k]:
                    diff[field] += 1
        print(f"{arm} vs baseline over {len(shared)} crops: "
              f"date differs {diff['verbatimDate']}, locality differs {diff['verbatimLocality']}")